# 03. Readable code

**Tier:** Core. **Why you'd use this:** You hand a script to a teammate and they struggle to read it, or a linter fills your screen with warnings. Python has an official style guide, PEP 8, and `ruff` can check most of it for you.

**After this lesson you can:** name variables, functions, classes, and constants the PEP 8 way; read a ruff naming or layout code; fix the code it flags.
About 20 minutes. **Uses:** [Tooling](01-tooling-lesson.ipynb), [Docstrings and hints](../05-functions/04-hints-lesson.ipynb)

## Quick review

1. What can ruff catch without running your program?
2. Does a wrong type hint stop a call at runtime?

<details><summary>Answers</summary>

Style problems and likely bugs, such as an unused variable. No, hints are ignored when the code runs.
</details>

## The idea

PEP 8 is Python's style guide. It asks for `snake_case` for functions and variables, `CapWords` for classes, and `UPPER_SNAKE_CASE` for constants. It also asks for four-space indents, imports at the top, and lines that fit on the screen. None of this changes how code runs. It changes how fast a person can read it.

Ruff checks the rules a tool can check. Each finding has a code. `N` codes cover names, `E` and `W` cover layout, `I` covers import order, and `F` covers likely bugs such as unused imports. This repo turns all of them on.

<details><summary>Explain more</summary>

A code like `N802` means "function name should be lowercase". Run `ruff rule N802` to read the full explanation. `ruff check --fix` repairs the findings that are safe to automate, such as import order. A rename is your call.
</details>

## See it work

Lint a messy snippet, read the codes, then lint the cleaned version.

In [1]:
import json
import subprocess
import sys

MESSY = """import os


def CalcTotal(Prices):
    Total = 0
    for p in Prices:
        Total += p
    return Total
"""

CLEAN = """def calc_total(prices):
    total = 0
    for price in prices:
        total += price
    return total
"""


def lint_codes(source: str) -> list[str]:
    # 1. Feed the source to ruff on stdin, so there is no file to clean up.
    command = [
        sys.executable,
        "-m",
        "ruff",
        "check",
        "--isolated",
        "--select",
        "N,E,F,I",
        "--output-format",
        "json",
        "--stdin-filename",
        "sample.py",
        "-",
    ]
    result = subprocess.run(command, input=source, capture_output=True, text=True)
    # 2. JSON output has no colors, so it parses the same in a notebook and a shell.
    return sorted({finding["code"] for finding in json.loads(result.stdout)})


print(lint_codes(MESSY))
print(lint_codes(CLEAN))

['F401', 'N802', 'N803', 'N806']
[]


## Check yourself

Answer these before you run the cell.

1. Which code flags the name `CalcTotal`?
2. Which code flags `import os`?
3. Does Python refuse to run `MESSY`?

In [2]:
namespace = {}
exec(MESSY, namespace)  # run the string as a program
print(namespace["CalcTotal"]([1, 2, 3]))

6


<details><summary>Check your answers</summary>

`N802` (function name) and `F401` (unused import). No: it runs and prints `6`. Style rules never change behavior.
</details>

## Why it works this way

Style rules exist because code is read far more often than it is written. A shared convention lets a reader tell a class from a function from a constant by its spelling alone, before they read a line. `CapWords` for classes and `snake_case` for functions are the standard library's own spelling, so your code looks like the code it calls.

Ruff is static: it parses the text and never runs it. That makes it fast and safe, and it also means it can only judge spelling, layout, and patterns it recognizes. It cannot tell you a name is vague. `d` passes every rule and still tells the reader nothing. Deep dive on the tool itself: [Tooling](01-tooling-lesson.ipynb).

The cost of ignoring a convention is small per line and large across a project, because every reader pays to translate. The one fair exception is a name that must match the outside world, such as a JSON key or a method you override. Keep the outside spelling there, and keep your own names PEP 8.

In [3]:
print(lint_codes("def f(d):\n    return d\n"))

[]


## When to use it

Follow PEP 8 for every name you choose. Run `ruff check` before you share code, and read the code it gives you before you fix. Keep an outside name when you must match one. Use `ruff check --fix` only on code saved in version control, so you can review the change. Don't rename something only to silence a warning you have not read, and don't treat a clean ruff run as proof that names are good.

## Common mistake

Writing `camelCase` names out of habit from another language. Python runs them without complaint, so only a linter or a reviewer tells you.

In [4]:
def calcTotal(prices):
    return sum(prices)


print(calcTotal([1, 2, 3]))

6


It runs and prints `6`, but ruff reports `N802`. Rename it to `calc_total`.

In [5]:
def calc_total(prices):
    return sum(prices)


print(calc_total([1, 2, 3]))
print(lint_codes("def calc_total(prices):\n    return sum(prices)\n"))

6
[]


## Try it

Add `import json` to the top of `CLEAN` and run `lint_codes` again. Which code appears, and what is the one-line fix?

## Recap

- PEP 8: `snake_case` for functions and variables, `CapWords` for classes, `UPPER_SNAKE_CASE` for constants.
- Ruff codes group by letter: `N` names, `E` and `W` layout, `I` imports, `F` likely bugs.
- Style rules change readability, never behavior.
- A clean lint run does not make a name clear.

**Next:** [practice](03-readable-code-practice.ipynb)